# Autoencoders: A Comprehensive Guide

This notebook provides a comprehensive introduction to autoencoders, including theory, implementation from scratch, and practical applications.

---
## 1. Theory Section

### 1.1 What is an Autoencoder?

An **autoencoder** is a type of neural network that learns to compress data into a lower-dimensional representation and then reconstruct it back to the original form. The network consists of two main parts:

1. **Encoder**: Maps input data $\mathbf{x}$ to a latent representation $\mathbf{z}$
2. **Decoder**: Reconstructs the input from the latent representation $\hat{\mathbf{x}}$

### 1.2 Encoder-Decoder Architecture

```
Input (784) -> [Encoder] -> Latent Space (e.g., 2-32) -> [Decoder] -> Output (784)
    x                              z                              x_hat
```

**Mathematical Formulation:**
- Encoder: $\mathbf{z} = f_\theta(\mathbf{x}) = \sigma(\mathbf{W}_e \mathbf{x} + \mathbf{b}_e)$
- Decoder: $\hat{\mathbf{x}} = g_\phi(\mathbf{z}) = \sigma(\mathbf{W}_d \mathbf{z} + \mathbf{b}_d)$

where $\sigma$ is an activation function (ReLU, sigmoid, etc.)

### 1.3 Bottleneck Layer

The **bottleneck** (latent layer) is the layer with the smallest number of neurons. It forces the network to:
- Learn compressed representations
- Capture the most important features
- Discard noise and redundant information

The dimensionality of the bottleneck determines the compression ratio.

### 1.4 Reconstruction Loss

The training objective is to minimize the **reconstruction loss**:

$$\mathcal{L}(\mathbf{x}, \hat{\mathbf{x}}) = \|\mathbf{x} - \hat{\mathbf{x}}\|^2 = \sum_{i=1}^{n}(x_i - \hat{x}_i)^2$$

This is the Mean Squared Error (MSE) between input and reconstruction.

### 1.5 Latent Space

The **latent space** is the space of encoded representations. Good autoencoders learn:
- Meaningful representations where similar inputs map to nearby points
- Smooth interpolations between data points
- Disentangled features (in well-designed architectures)

### 1.6 Comparison with PCA

| Aspect | PCA | Autoencoder |
|--------|-----|-------------|
| **Transformation** | Linear | Non-linear |
| **Optimization** | Closed-form (SVD) | Gradient descent |
| **Flexibility** | Fixed structure | Customizable architecture |
| **Computational Cost** | Lower | Higher |
| **Interpretability** | High (principal components) | Lower |
| **Data Requirements** | Works with small data | Needs more data |

**Key Insight**: A linear autoencoder with MSE loss learns the same subspace as PCA. The power of autoencoders comes from non-linear activations.

### 1.7 Autoencoder Variants (Brief Overview)

**1. Denoising Autoencoder (DAE)**
- Input is corrupted with noise
- Network learns to reconstruct clean input
- Learns more robust features

**2. Sparse Autoencoder**
- Adds sparsity constraint to latent layer
- Loss: $\mathcal{L}_{total} = \mathcal{L}_{recon} + \lambda \|\mathbf{z}\|_1$
- Learns more interpretable features

**3. Variational Autoencoder (VAE)**
- Learns a probability distribution in latent space
- Encoder outputs mean and variance
- Enables generation of new samples
- Loss: $\mathcal{L} = \mathcal{L}_{recon} + D_{KL}(q(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}))$

---
## 2. Implementation from Scratch

We'll implement a fully-connected autoencoder using only NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
class Autoencoder:
    """
    A simple fully-connected autoencoder implemented with NumPy.
    
    Architecture: input -> encoder layers -> latent -> decoder layers -> output
    
    Parameters
    ----------
    input_dim : int
        Dimension of input data
    hidden_dims : list of int
        Dimensions of hidden layers in encoder (decoder mirrors this)
    latent_dim : int
        Dimension of the latent (bottleneck) layer
    activation : str
        Activation function: 'relu', 'sigmoid', or 'tanh'
    learning_rate : float
        Learning rate for gradient descent
    """
    
    def __init__(self, input_dim, hidden_dims=[64, 32], latent_dim=2,
                 activation='relu', learning_rate=0.001):
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.latent_dim = latent_dim
        self.learning_rate = learning_rate
        self.activation_name = activation
        
        # Build layer dimensions
        # Encoder: input -> hidden_dims -> latent
        # Decoder: latent -> hidden_dims (reversed) -> input
        self.encoder_dims = [input_dim] + hidden_dims + [latent_dim]
        self.decoder_dims = [latent_dim] + hidden_dims[::-1] + [input_dim]
        
        # Initialize weights and biases
        self.encoder_weights = []
        self.encoder_biases = []
        self.decoder_weights = []
        self.decoder_biases = []
        
        self._initialize_weights()
        
        # Training history
        self.history = {'loss': [], 'val_loss': []}
    
    def _initialize_weights(self):
        """Initialize weights using He initialization for ReLU."""
        # Encoder weights
        for i in range(len(self.encoder_dims) - 1):
            fan_in = self.encoder_dims[i]
            fan_out = self.encoder_dims[i + 1]
            # He initialization
            std = np.sqrt(2.0 / fan_in)
            W = np.random.randn(fan_in, fan_out) * std
            b = np.zeros((1, fan_out))
            self.encoder_weights.append(W)
            self.encoder_biases.append(b)
        
        # Decoder weights
        for i in range(len(self.decoder_dims) - 1):
            fan_in = self.decoder_dims[i]
            fan_out = self.decoder_dims[i + 1]
            std = np.sqrt(2.0 / fan_in)
            W = np.random.randn(fan_in, fan_out) * std
            b = np.zeros((1, fan_out))
            self.decoder_weights.append(W)
            self.decoder_biases.append(b)
    
    def _activation(self, x):
        """Apply activation function."""
        if self.activation_name == 'relu':
            return np.maximum(0, x)
        elif self.activation_name == 'sigmoid':
            return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        elif self.activation_name == 'tanh':
            return np.tanh(x)
        else:
            raise ValueError(f"Unknown activation: {self.activation_name}")
    
    def _activation_derivative(self, x):
        """Compute derivative of activation function."""
        if self.activation_name == 'relu':
            return (x > 0).astype(float)
        elif self.activation_name == 'sigmoid':
            s = self._activation(x)
            return s * (1 - s)
        elif self.activation_name == 'tanh':
            return 1 - np.tanh(x) ** 2
    
    def encode(self, X):
        """
        Encode input data to latent representation.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, input_dim)
            Input data
            
        Returns
        -------
        z : ndarray of shape (n_samples, latent_dim)
            Latent representation
        """
        h = X
        for i, (W, b) in enumerate(zip(self.encoder_weights, self.encoder_biases)):
            z = h @ W + b
            # Apply activation to all layers except the last (latent layer)
            if i < len(self.encoder_weights) - 1:
                h = self._activation(z)
            else:
                h = z  # Linear activation for latent layer
        return h
    
    def decode(self, z):
        """
        Decode latent representation to reconstructed input.
        
        Parameters
        ----------
        z : ndarray of shape (n_samples, latent_dim)
            Latent representation
            
        Returns
        -------
        x_hat : ndarray of shape (n_samples, input_dim)
            Reconstructed data
        """
        h = z
        for i, (W, b) in enumerate(zip(self.decoder_weights, self.decoder_biases)):
            z_out = h @ W + b
            # Apply activation to all layers except the last (output layer)
            if i < len(self.decoder_weights) - 1:
                h = self._activation(z_out)
            else:
                # Sigmoid for output to match [0, 1] scaled input
                h = 1 / (1 + np.exp(-np.clip(z_out, -500, 500)))
        return h
    
    def reconstruct(self, X):
        """
        Reconstruct input data through encode-decode pipeline.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, input_dim)
            Input data
            
        Returns
        -------
        x_hat : ndarray of shape (n_samples, input_dim)
            Reconstructed data
        """
        z = self.encode(X)
        return self.decode(z)
    
    def _forward_pass(self, X):
        """
        Perform forward pass and cache activations for backprop.
        
        Returns
        -------
        encoder_cache : list of tuples (pre_activation, post_activation)
        decoder_cache : list of tuples (pre_activation, post_activation)
        x_hat : reconstructed output
        """
        encoder_cache = []
        h = X
        
        # Encoder forward
        for i, (W, b) in enumerate(zip(self.encoder_weights, self.encoder_biases)):
            z = h @ W + b
            if i < len(self.encoder_weights) - 1:
                h_new = self._activation(z)
            else:
                h_new = z  # Linear for latent
            encoder_cache.append((h, z, h_new))
            h = h_new
        
        latent = h
        decoder_cache = []
        
        # Decoder forward
        for i, (W, b) in enumerate(zip(self.decoder_weights, self.decoder_biases)):
            z = h @ W + b
            if i < len(self.decoder_weights) - 1:
                h_new = self._activation(z)
            else:
                h_new = 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # Sigmoid output
            decoder_cache.append((h, z, h_new))
            h = h_new
        
        return encoder_cache, decoder_cache, latent, h
    
    def _backward_pass(self, X, encoder_cache, decoder_cache, x_hat):
        """
        Perform backward pass to compute gradients.
        
        Returns
        -------
        encoder_grads : list of (dW, db) tuples
        decoder_grads : list of (dW, db) tuples
        """
        batch_size = X.shape[0]
        
        # Gradient of MSE loss: d/dx_hat (x - x_hat)^2 = -2(x - x_hat)
        # Combined with sigmoid derivative for output layer
        delta = -(X - x_hat) * x_hat * (1 - x_hat)  # Sigmoid derivative
        
        decoder_grads = []
        
        # Decoder backward
        for i in range(len(self.decoder_weights) - 1, -1, -1):
            h_prev, z, h = decoder_cache[i]
            
            dW = h_prev.T @ delta / batch_size
            db = np.mean(delta, axis=0, keepdims=True)
            decoder_grads.insert(0, (dW, db))
            
            if i > 0:
                delta = (delta @ self.decoder_weights[i].T) * self._activation_derivative(decoder_cache[i-1][1])
            else:
                # Pass gradient to encoder (latent layer is linear)
                delta = delta @ self.decoder_weights[i].T
        
        encoder_grads = []
        
        # Encoder backward
        for i in range(len(self.encoder_weights) - 1, -1, -1):
            h_prev, z, h = encoder_cache[i]
            
            # For latent layer (last encoder layer), no activation derivative
            if i < len(self.encoder_weights) - 1:
                delta = delta * self._activation_derivative(z)
            
            dW = h_prev.T @ delta / batch_size
            db = np.mean(delta, axis=0, keepdims=True)
            encoder_grads.insert(0, (dW, db))
            
            if i > 0:
                delta = delta @ self.encoder_weights[i].T
        
        return encoder_grads, decoder_grads
    
    def _update_weights(self, encoder_grads, decoder_grads):
        """Update weights using gradient descent."""
        for i, (dW, db) in enumerate(encoder_grads):
            self.encoder_weights[i] -= self.learning_rate * dW
            self.encoder_biases[i] -= self.learning_rate * db
        
        for i, (dW, db) in enumerate(decoder_grads):
            self.decoder_weights[i] -= self.learning_rate * dW
            self.decoder_biases[i] -= self.learning_rate * db
    
    def _compute_loss(self, X, x_hat):
        """Compute Mean Squared Error loss."""
        return np.mean((X - x_hat) ** 2)
    
    def fit(self, X, X_val=None, epochs=100, batch_size=32, verbose=True):
        """
        Train the autoencoder.
        
        Parameters
        ----------
        X : ndarray of shape (n_samples, input_dim)
            Training data
        X_val : ndarray of shape (n_val_samples, input_dim), optional
            Validation data
        epochs : int
            Number of training epochs
        batch_size : int
            Mini-batch size
        verbose : bool
            Whether to print training progress
            
        Returns
        -------
        self : Autoencoder
            Trained autoencoder
        """
        n_samples = X.shape[0]
        n_batches = int(np.ceil(n_samples / batch_size))
        
        for epoch in range(epochs):
            # Shuffle data
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            
            epoch_loss = 0
            
            for batch_idx in range(n_batches):
                start_idx = batch_idx * batch_size
                end_idx = min(start_idx + batch_size, n_samples)
                X_batch = X_shuffled[start_idx:end_idx]
                
                # Forward pass
                encoder_cache, decoder_cache, latent, x_hat = self._forward_pass(X_batch)
                
                # Compute loss
                batch_loss = self._compute_loss(X_batch, x_hat)
                epoch_loss += batch_loss * (end_idx - start_idx)
                
                # Backward pass
                encoder_grads, decoder_grads = self._backward_pass(
                    X_batch, encoder_cache, decoder_cache, x_hat
                )
                
                # Update weights
                self._update_weights(encoder_grads, decoder_grads)
            
            # Record training loss
            epoch_loss /= n_samples
            self.history['loss'].append(epoch_loss)
            
            # Compute validation loss
            if X_val is not None:
                x_val_hat = self.reconstruct(X_val)
                val_loss = self._compute_loss(X_val, x_val_hat)
                self.history['val_loss'].append(val_loss)
            
            # Print progress
            if verbose and (epoch + 1) % 10 == 0:
                msg = f"Epoch {epoch + 1}/{epochs} - Loss: {epoch_loss:.6f}"
                if X_val is not None:
                    msg += f" - Val Loss: {val_loss:.6f}"
                print(msg)
        
        return self

---
## 3. Training & Optimization

We'll use the sklearn digits dataset (8x8 grayscale images of digits 0-9).

In [ ]:
# Load and preprocess data
digits = load_digits()
X = digits.data  # Shape: (1797, 64)
y = digits.target

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Pixel value range: [{X.min():.1f}, {X.max():.1f}]")

# Scale to [0, 1]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split into train/validation/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nTrain: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# Create and train autoencoder
# Architecture: 64 -> 32 -> 16 -> 2 -> 16 -> 32 -> 64
autoencoder = Autoencoder(
    input_dim=64,
    hidden_dims=[32, 16],
    latent_dim=2,
    activation='relu',
    learning_rate=0.01
)

print("Training autoencoder...")
print(f"Architecture: {autoencoder.encoder_dims} -> {autoencoder.decoder_dims}")
print()

autoencoder.fit(
    X_train,
    X_val=X_val,
    epochs=200,
    batch_size=32,
    verbose=True
)

---
## 4. Diagnostics & Evaluation

In [ ]:
# Evaluate reconstruction quality
def evaluate_reconstruction(model, X, name=""):
    """Evaluate reconstruction quality metrics."""
    X_recon = model.reconstruct(X)
    
    # MSE
    mse = np.mean((X - X_recon) ** 2)
    
    # RMSE
    rmse = np.sqrt(mse)
    
    # Mean Absolute Error
    mae = np.mean(np.abs(X - X_recon))
    
    # R-squared (coefficient of determination)
    ss_res = np.sum((X - X_recon) ** 2)
    ss_tot = np.sum((X - np.mean(X)) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    print(f"{name} Reconstruction Metrics:")
    print(f"  MSE:  {mse:.6f}")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  MAE:  {mae:.6f}")
    print(f"  R^2:  {r2:.4f}")
    
    return mse, rmse, mae, r2

print("Autoencoder Performance:")
print("=" * 40)
train_metrics = evaluate_reconstruction(autoencoder, X_train, "Train")
print()
val_metrics = evaluate_reconstruction(autoencoder, X_val, "Validation")
print()
test_metrics = evaluate_reconstruction(autoencoder, X_test, "Test")

In [ ]:
# Analyze latent space statistics
z_train = autoencoder.encode(X_train)
z_test = autoencoder.encode(X_test)

print("Latent Space Statistics:")
print("=" * 40)
print(f"Latent dimension: {z_train.shape[1]}")
print(f"\nTrain latent space:")
print(f"  Mean: [{z_train[:, 0].mean():.3f}, {z_train[:, 1].mean():.3f}]")
print(f"  Std:  [{z_train[:, 0].std():.3f}, {z_train[:, 1].std():.3f}]")
print(f"  Range Z1: [{z_train[:, 0].min():.3f}, {z_train[:, 0].max():.3f}]")
print(f"  Range Z2: [{z_train[:, 1].min():.3f}, {z_train[:, 1].max():.3f}]")

---
## 5. Visualizations

In [ ]:
# Plot training history (loss curves)
fig, ax = plt.subplots(figsize=(10, 5))

epochs_range = range(1, len(autoencoder.history['loss']) + 1)
ax.plot(epochs_range, autoencoder.history['loss'], 'b-', label='Training Loss', linewidth=2)
ax.plot(epochs_range, autoencoder.history['val_loss'], 'r--', label='Validation Loss', linewidth=2)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Autoencoder Training History', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize original vs reconstructed images
def plot_reconstructions(model, X, y, n_samples=10, title=""):
    """Plot original images and their reconstructions."""
    # Select random samples
    indices = np.random.choice(len(X), n_samples, replace=False)
    X_sample = X[indices]
    y_sample = y[indices]
    
    # Reconstruct
    X_recon = model.reconstruct(X_sample)
    
    fig, axes = plt.subplots(3, n_samples, figsize=(15, 5))
    
    for i in range(n_samples):
        # Original
        axes[0, i].imshow(X_sample[i].reshape(8, 8), cmap='gray')
        axes[0, i].axis('off')
        axes[0, i].set_title(f'Label: {y_sample[i]}', fontsize=9)
        
        # Reconstructed
        axes[1, i].imshow(X_recon[i].reshape(8, 8), cmap='gray')
        axes[1, i].axis('off')
        
        # Difference
        diff = np.abs(X_sample[i] - X_recon[i])
        axes[2, i].imshow(diff.reshape(8, 8), cmap='hot')
        axes[2, i].axis('off')
    
    axes[0, 0].set_ylabel('Original', fontsize=11)
    axes[1, 0].set_ylabel('Reconstructed', fontsize=11)
    axes[2, 0].set_ylabel('Difference', fontsize=11)
    
    fig.suptitle(f'{title} - Original vs Reconstructed', fontsize=14)
    plt.tight_layout()
    plt.show()

np.random.seed(123)
plot_reconstructions(autoencoder, X_test, y_test, n_samples=10, title="Autoencoder")

In [ ]:
# Visualize latent space (2D)
def plot_latent_space(model, X, y, title="Latent Space"):
    """Plot 2D latent space colored by digit labels."""
    z = model.encode(X)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    scatter = ax.scatter(z[:, 0], z[:, 1], c=y, cmap='tab10', 
                         alpha=0.7, s=30, edgecolors='white', linewidth=0.5)
    
    # Add colorbar with digit labels
    cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
    cbar.set_label('Digit', fontsize=12)
    
    ax.set_xlabel('Latent Dimension 1', fontsize=12)
    ax.set_ylabel('Latent Dimension 2', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return z

z_all = plot_latent_space(autoencoder, X_scaled, y, title="Autoencoder Latent Space (2D)")

In [ ]:
# Latent space interpolation
def plot_latent_interpolation(model, X, y, digit1, digit2, n_steps=10):
    """Interpolate between two digits in latent space."""
    # Find examples of each digit
    idx1 = np.where(y == digit1)[0][0]
    idx2 = np.where(y == digit2)[0][0]
    
    # Encode
    z1 = model.encode(X[idx1:idx1+1])
    z2 = model.encode(X[idx2:idx2+1])
    
    # Interpolate
    alphas = np.linspace(0, 1, n_steps)
    z_interp = np.array([z1 * (1 - a) + z2 * a for a in alphas]).squeeze()
    
    # Decode
    x_interp = model.decode(z_interp)
    
    # Plot
    fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))
    for i, ax in enumerate(axes):
        ax.imshow(x_interp[i].reshape(8, 8), cmap='gray')
        ax.axis('off')
        ax.set_title(f'{alphas[i]:.1f}', fontsize=9)
    
    fig.suptitle(f'Latent Space Interpolation: {digit1} -> {digit2}', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_latent_interpolation(autoencoder, X_scaled, y, digit1=3, digit2=8, n_steps=10)
plot_latent_interpolation(autoencoder, X_scaled, y, digit1=0, digit2=1, n_steps=10)

In [ ]:
# Generate grid of decoded images from latent space
def plot_latent_grid(model, X, n_grid=15):
    """Generate a grid of images by sampling the latent space."""
    # Get latent space bounds from data
    z = model.encode(X)
    z1_range = np.linspace(np.percentile(z[:, 0], 5), np.percentile(z[:, 0], 95), n_grid)
    z2_range = np.linspace(np.percentile(z[:, 1], 5), np.percentile(z[:, 1], 95), n_grid)
    
    # Create grid
    grid_images = []
    for z2 in reversed(z2_range):
        row = []
        for z1 in z1_range:
            z_point = np.array([[z1, z2]])
            decoded = model.decode(z_point)
            row.append(decoded.reshape(8, 8))
        grid_images.append(row)
    
    # Plot
    fig, axes = plt.subplots(n_grid, n_grid, figsize=(12, 12))
    for i in range(n_grid):
        for j in range(n_grid):
            axes[i, j].imshow(grid_images[i][j], cmap='gray')
            axes[i, j].axis('off')
    
    fig.suptitle('Decoded Images from Latent Space Grid', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_latent_grid(autoencoder, X_scaled, n_grid=12)

---
## 6. Use Cases & Guidelines

### 6.1 When to Use Autoencoders

**Non-linear Dimensionality Reduction**
- When data has complex, non-linear relationships
- PCA or linear methods underperform
- Need to preserve local structure

**Feature Learning**
- Pre-training for downstream tasks
- Learning representations without labels
- Transfer learning scenarios

**Anomaly Detection**
- Train on normal data only
- High reconstruction error indicates anomaly
- Works well for fraud detection, defect detection

**Data Denoising**
- Denoising autoencoders learn to remove noise
- Useful for image/signal cleaning

**Data Generation** (VAE)
- Generate new samples similar to training data
- Image synthesis, data augmentation

### 6.2 When NOT to Use Autoencoders

**Small Datasets**
- Neural networks need substantial data
- Use PCA or kernel PCA instead
- Risk of overfitting

**Interpretability Required**
- Latent dimensions are not interpretable
- PCA components have clear meanings
- Regulatory requirements may mandate explainability

**Linear Relationships Suffice**
- If PCA works well, stick with it
- Simpler is better (Occam's Razor)
- Lower computational cost

**Real-time Inference Constraints**
- Neural network inference can be slow
- Consider model compression techniques
- PCA is faster for simple projections

### 6.3 Architecture Selection Tips

| Factor | Recommendation |
|--------|----------------|
| **Latent Dimension** | Start with 2D for visualization; increase until reconstruction quality plateaus |
| **Hidden Layers** | 2-4 layers typically sufficient; symmetric encoder/decoder |
| **Layer Width** | Gradual reduction (e.g., 512->256->128->latent) |
| **Activation** | ReLU for hidden layers; Sigmoid for output if input is [0,1] |
| **Learning Rate** | Start with 0.001; use learning rate schedulers |
| **Batch Size** | 32-256; smaller batches for more noise in gradient |
| **Regularization** | Dropout (0.1-0.3), L2 weight decay, early stopping |

---
## 7. Comparison with sklearn/PCA

Let's compare our autoencoder with PCA at the same dimensionality.

In [ ]:
# Train PCA with same latent dimension
pca = PCA(n_components=2)
z_pca_train = pca.fit_transform(X_train)
z_pca_test = pca.transform(X_test)

# Reconstruct using PCA
X_pca_recon_train = pca.inverse_transform(z_pca_train)
X_pca_recon_test = pca.inverse_transform(z_pca_test)

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"PCA total variance explained: {pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
# Compare reconstruction metrics
print("Reconstruction Comparison (Test Set):")
print("=" * 50)

# Autoencoder
X_ae_recon_test = autoencoder.reconstruct(X_test)
ae_mse = np.mean((X_test - X_ae_recon_test) ** 2)
ae_r2 = 1 - np.sum((X_test - X_ae_recon_test) ** 2) / np.sum((X_test - X_test.mean()) ** 2)

# PCA
pca_mse = np.mean((X_test - X_pca_recon_test) ** 2)
pca_r2 = 1 - np.sum((X_test - X_pca_recon_test) ** 2) / np.sum((X_test - X_test.mean()) ** 2)

print(f"\n{'Method':<20} {'MSE':<12} {'R-squared':<12}")
print("-" * 44)
print(f"{'Autoencoder':<20} {ae_mse:<12.6f} {ae_r2:<12.4f}")
print(f"{'PCA':<20} {pca_mse:<12.6f} {pca_r2:<12.4f}")
print("-" * 44)
print(f"\nAutoencoder MSE improvement: {(pca_mse - ae_mse) / pca_mse * 100:.1f}%")

In [ ]:
# Visual comparison of reconstructions
def compare_reconstructions(X, y, ae_model, pca_model, n_samples=8):
    """Compare reconstructions from autoencoder and PCA."""
    indices = np.random.choice(len(X), n_samples, replace=False)
    X_sample = X[indices]
    y_sample = y[indices]
    
    # Reconstruct
    X_ae_recon = ae_model.reconstruct(X_sample)
    X_pca_recon = pca_model.inverse_transform(pca_model.transform(X_sample))
    
    fig, axes = plt.subplots(3, n_samples, figsize=(14, 5))
    
    for i in range(n_samples):
        # Original
        axes[0, i].imshow(X_sample[i].reshape(8, 8), cmap='gray')
        axes[0, i].axis('off')
        axes[0, i].set_title(f'{y_sample[i]}', fontsize=10)
        
        # Autoencoder
        axes[1, i].imshow(X_ae_recon[i].reshape(8, 8), cmap='gray')
        axes[1, i].axis('off')
        
        # PCA
        axes[2, i].imshow(X_pca_recon[i].reshape(8, 8), cmap='gray')
        axes[2, i].axis('off')
    
    axes[0, 0].set_ylabel('Original', fontsize=11)
    axes[1, 0].set_ylabel('Autoencoder', fontsize=11)
    axes[2, 0].set_ylabel('PCA', fontsize=11)
    
    fig.suptitle('Reconstruction Comparison: Autoencoder vs PCA (2D latent)', fontsize=14)
    plt.tight_layout()
    plt.show()

np.random.seed(456)
compare_reconstructions(X_test, y_test, autoencoder, pca, n_samples=10)

In [ ]:
# Compare latent space visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Autoencoder latent space
z_ae = autoencoder.encode(X_scaled)
scatter1 = axes[0].scatter(z_ae[:, 0], z_ae[:, 1], c=y, cmap='tab10', 
                           alpha=0.7, s=20, edgecolors='white', linewidth=0.3)
axes[0].set_xlabel('Latent Dimension 1', fontsize=11)
axes[0].set_ylabel('Latent Dimension 2', fontsize=11)
axes[0].set_title('Autoencoder Latent Space', fontsize=13)
axes[0].grid(True, alpha=0.3)

# PCA latent space
z_pca = pca.transform(X_scaled)
scatter2 = axes[1].scatter(z_pca[:, 0], z_pca[:, 1], c=y, cmap='tab10', 
                           alpha=0.7, s=20, edgecolors='white', linewidth=0.3)
axes[1].set_xlabel('PC 1', fontsize=11)
axes[1].set_ylabel('PC 2', fontsize=11)
axes[1].set_title('PCA Latent Space', fontsize=13)
axes[1].grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter2, ax=axes, ticks=range(10), shrink=0.8)
cbar.set_label('Digit', fontsize=11)

plt.suptitle('Latent Space Comparison: Autoencoder vs PCA', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compare reconstruction quality at different latent dimensions
def compare_dimensions(X_train, X_test, dimensions=[2, 4, 8, 16, 32]):
    """Compare AE and PCA at different latent dimensions."""
    results = {'dim': [], 'ae_mse': [], 'pca_mse': []}
    
    for dim in dimensions:
        print(f"\nEvaluating latent_dim={dim}...")
        
        # Train autoencoder
        ae = Autoencoder(
            input_dim=64,
            hidden_dims=[32, 16],
            latent_dim=dim,
            activation='relu',
            learning_rate=0.01
        )
        ae.fit(X_train, epochs=100, batch_size=32, verbose=False)
        
        # Train PCA
        pca_model = PCA(n_components=dim)
        pca_model.fit(X_train)
        
        # Compute MSE
        ae_recon = ae.reconstruct(X_test)
        pca_recon = pca_model.inverse_transform(pca_model.transform(X_test))
        
        ae_mse = np.mean((X_test - ae_recon) ** 2)
        pca_mse = np.mean((X_test - pca_recon) ** 2)
        
        results['dim'].append(dim)
        results['ae_mse'].append(ae_mse)
        results['pca_mse'].append(pca_mse)
        
        print(f"  AE MSE: {ae_mse:.6f}, PCA MSE: {pca_mse:.6f}")
    
    return results

print("Comparing reconstruction quality across latent dimensions...")
comparison_results = compare_dimensions(X_train, X_test, dimensions=[2, 4, 8, 16, 32])

In [ ]:
# Plot comparison results
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(comparison_results['dim'], comparison_results['ae_mse'], 
        'b-o', label='Autoencoder', linewidth=2, markersize=8)
ax.plot(comparison_results['dim'], comparison_results['pca_mse'], 
        'r--s', label='PCA', linewidth=2, markersize=8)

ax.set_xlabel('Latent Dimension', fontsize=12)
ax.set_ylabel('Test MSE', fontsize=12)
ax.set_title('Reconstruction Quality: Autoencoder vs PCA', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(comparison_results['dim'])

plt.tight_layout()
plt.show()

# Print summary table
print("\nSummary Table:")
print("=" * 50)
print(f"{'Latent Dim':<12} {'AE MSE':<12} {'PCA MSE':<12} {'Improvement':<12}")
print("-" * 50)
for i in range(len(comparison_results['dim'])):
    dim = comparison_results['dim'][i]
    ae_mse = comparison_results['ae_mse'][i]
    pca_mse = comparison_results['pca_mse'][i]
    improvement = (pca_mse - ae_mse) / pca_mse * 100
    print(f"{dim:<12} {ae_mse:<12.6f} {pca_mse:<12.6f} {improvement:>+.1f}%")

---
## 8. Summary

### Key Takeaways

1. **Autoencoders** are neural networks that learn compressed representations through reconstruction.

2. **Architecture**: Encoder compresses input to latent space; decoder reconstructs from latent space.

3. **Non-linear advantage**: Unlike PCA, autoencoders can capture non-linear relationships.

4. **Trade-offs**:
   - More flexible but less interpretable than PCA
   - Require more data and computation
   - Better reconstruction at low dimensions

5. **Applications**: Dimensionality reduction, feature learning, anomaly detection, denoising.

### Next Steps

- Experiment with deeper architectures
- Try convolutional autoencoders for image data
- Implement variational autoencoders (VAEs) for generation
- Apply to anomaly detection tasks